# Ensemble tests

Runs every ranker configuration in `CONFIGS` on every dataset in `DATASETS`, evaluates each run against `datasets/<name>/qrels/<QRELS_SPLIT>.tsv`, and collects the metrics in one table.

Vectors must already exist (`python src/encode.py --method <encoder>`). Use the **aws** kernel.

In [10]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from search import run_search

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configure

Each entry in `CONFIGS` is `(label, ranker, options)`:
- `"chamfer"` takes `symmetric`.
- `"sw"` (sliced Wasserstein) takes `slices`, `quantiles`, `p`, `seed`. Slices are nested, so for one seed a larger `slices` extends a smaller one with more directions.

`IGNORE_IDENTICAL_IDS` lists datasets whose queries also appear in the corpus (BEIR convention: drop the self-match).

In [19]:
DATASETS = ["arguana", "scidocs", "quora"]
ENCODER = "bert"
QUERY_TAG = None      # only needed when a dataset has several vector sets for ENCODER
CORPUS_TAG = None
K = 100
QRELS_SPLIT = "test"
METRICS = ("ndcg@10", "recall@100", "mrr@10")
IGNORE_IDENTICAL_IDS = {"arguana"}
WRITE = True          # also save each run to results/<dataset>/<encoder>/

CONFIGS = [
    ("chamfer",     "chamfer", {}),
    ("chamfer-sym", "chamfer", {"symmetric": True}),
] + [
    (f"sw L={L}",   "sw",      {"slices": L, "quantiles": 32, "p": 2, "seed": 0})
    for L in (64, 128, 256, 512, 1024, 2048)
]

## Run

In [20]:
rows = []
for dataset in DATASETS:
    for label, ranker, opts in CONFIGS:
        out = run_search(dataset, ranker, ENCODER, QUERY_TAG, CORPUS_TAG, k=K,
                         ignore_identical_ids=dataset in IGNORE_IDENTICAL_IDS, write=WRITE,
                         qrels_split=QRELS_SPLIT, metrics=METRICS, **opts)
        meta = out["meta"]
        if "metrics" not in meta:
            print(f"  no qrels for {dataset}/{QRELS_SPLIT}; skipping evaluation")
        rows.append({"dataset": dataset, "config": label, **meta.get("metrics", {}),
                     "seconds": meta["seconds"]})

[arguana] chamfer: 1406 queries x 8674 docs  device=cuda dtype=torch.float16
  8674/8674 docs  (2s)
  ndcg@10=0.3414  recall@100=0.8208  mrr@10=0.2791
  wrote results/arguana/bert/chamfer__bert-base-uncased_d768_nv32__bert-base-uncased_d768_nv180.trec  (2s)
[arguana] chamfer-sym: 1406 queries x 8674 docs  device=cuda dtype=torch.float16
  8674/8674 docs  (2s)
  ndcg@10=0.3435  recall@100=0.8201  mrr@10=0.2825
  wrote results/arguana/bert/chamfer-sym__bert-base-uncased_d768_nv32__bert-base-uncased_d768_nv180.trec  (3s)
[arguana] sw_L64_K32_p2_s0: 1406 queries x 8674 docs  device=cuda dtype=torch.float16
  8674/8674 docs  (1s)
  ndcg@10=0.1159  recall@100=0.4950  mrr@10=0.0885
  wrote results/arguana/bert/sw_L64_K32_p2_s0__bert-base-uncased_d768_nv32__bert-base-uncased_d768_nv180.trec  (1s)
[arguana] sw_L128_K32_p2_s0: 1406 queries x 8674 docs  device=cuda dtype=torch.float16
  8674/8674 docs  (1s)
  ndcg@10=0.1754  recall@100=0.6060  mrr@10=0.1349
  wrote results/arguana/bert/sw_L128_K3

## Results

In [21]:
table = pd.DataFrame(rows)
table

,dataset,config,ndcg@10,recall@100,mrr@10,seconds
0,arguana,chamfer,0.3414,0.8208,0.2791,1.7
1,arguana,chamfer-sym,0.3435,0.8201,0.2825,2.6
2,arguana,sw L=64,0.1159,0.4950,0.0885,0.6
3,arguana,sw L=128,0.1754,0.6060,0.1349,0.6
4,arguana,sw L=256,0.2089,0.6465,0.1641,0.6
5,arguana,sw L=512,0.2268,0.6764,0.1785,0.7
6,arguana,sw L=1024,0.2409,0.6871,0.1902,0.9
7,arguana,sw L=2048,0.2424,0.6977,0.1922,1.2
8,scidocs,chamfer,0.1043,0.2636,0.2000,56.6
9,scidocs,chamfer-sym,0.0596,0.2319,0.1086,5.0


In [22]:
# Side by side: one row per config, one column group per metric and dataset.
table.pivot(index="config", columns="dataset", values=list(METRICS)).reindex([c[0] for c in CONFIGS])

ndcg@10                 recall@100                  mrr@10  \
dataset     arguana   quora scidocs    arguana   quora scidocs arguana   
config                                                                   
chamfer      0.3414  0.6582  0.1043     0.8208  0.8975  0.2636  0.2791   
chamfer-sym  0.3435  0.6593  0.0596     0.8201  0.9039  0.2319  0.2825   
sw L=64      0.1159  0.5442  0.0162     0.4950  0.7904  0.0920  0.0885   
sw L=128     0.1754  0.5811  0.0269     0.6060  0.8315  0.1187  0.1349   
sw L=256     0.2089  0.5916  0.0286     0.6465  0.8432  0.1377  0.1641   
sw L=512     0.2268  0.6017  0.0327     0.6764  0.8534  0.1468  0.1785   
sw L=1024    0.2409  0.6075  0.0359     0.6871  0.8583  0.1539  0.1902   
sw L=2048    0.2424  0.6090  0.0351     0.6977  0.8601  0.1570  0.1922   

                             
dataset       quora scidocs  
config                       
chamfer      0.6543  0.2000  
chamfer-sym  0.6560  0.1086  
sw L=64      0.5434  0.0320  
sw L=128     0.5771  0.0498  
sw L=256     0.5867  0.0558  
sw L=512     0.5973  0.0599  
sw L=1024    0.6029  0.0675  
sw L=2048    0.6039  0.0645

In [ ]:
# table.to_csv("../results/ensemble.csv", index=False)